# PyTorch Exercises: From Basics to Custom Backprop

**TIFR PreSchool 2026 — ML School**

This notebook is a graded set of exercises. Each section introduces a concept and then asks you to fill in the empty solution cell that follows.

**How to use this notebook**
- Run the *setup* cell first.
- Read the prompt, then write your answer in the cell marked `# Your code here`.
- Some exercises include `assert` statements — your solution is (probably) correct when they pass silently.
- If you get stuck, look up the relevant function in the [PyTorch docs](https://pytorch.org/docs/stable/index.html).

**Outline**

1. Tensors and basic operations
2. Broadcasting, indexing, reshaping
3. Autograd — automatic differentiation
4. Building models with `nn.Module`
5. Datasets, `DataLoader`, and a training loop
6. Custom loss functions
7. Custom optimizers (writing your own SGD, Momentum, RMSProp, Adam)
8. Custom autograd functions (forward + backward by hand)
9. Hooks, gradient surgery, higher-order gradients
10. Putting it all together: a tiny end-to-end project

## Setup

Run this cell once at the start of the session.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

torch.manual_seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch version:', torch.__version__)
print('Device:', device)

---
## 1. Tensors and basic operations

A `torch.Tensor` is a multi-dimensional array with a `dtype`, a `device`, and (optionally) a `requires_grad` flag.

Useful constructors:
- `torch.tensor(data)` — from a Python list / NumPy array
- `torch.zeros`, `torch.ones`, `torch.eye`, `torch.arange`, `torch.linspace`
- `torch.randn`, `torch.rand`, `torch.randint`
- `.to(device)`, `.float()`, `.long()` for conversions

### Exercise 1.1 — Build a few tensors

Create the following tensors:

1. `a` — a `float32` tensor of shape `(3, 4)` filled with zeros.
2. `b` — the 5×5 identity matrix.
3. `c` — a tensor containing the integers 0, 2, 4, ..., 18 (i.e. even numbers from 0 up to 18 inclusive).
4. `d` — a `(2, 3, 4)` tensor of standard-normal random numbers.

Print each tensor's `shape` and `dtype`.

In [ ]:
a = torch.zeros((3, 4), dtype=torch.float32)
print("Tensor a:\n", a)
print("Shape:", a.shape, "| Dtype:", a.dtype, "\n")

b = torch.eye(5)
print("Tensor b:\n", b)
print("Shape:", b.shape, "\n")

c = torch.arange(0, 20, 2)
print("Tensor c:\n", c)
print("Shape:", c.shape, "\n")

d = torch.randn(2, 3, 4)
print("Tensor d:\n", d)
print("Shape:", d.shape)


### Exercise 1.2 — Element-wise math vs. matrix math

Given two random matrices `X` of shape `(4, 3)` and `Y` of shape `(3, 5)`:

1. Compute their **matrix product** `Z` of shape `(4, 5)` two different ways: with `@` and with `torch.matmul`.
2. Compute the element-wise product of `X` with itself.
3. Compute the sum of all elements of `Z`, the column-wise mean (one value per column), and the row-wise max.

Verify the two matrix-product results agree using `torch.allclose`.

In [ ]:
X = torch.randn(4, 3)
Y = torch.randn(3, 5)

Z_at = X @ Y
Z_matmul = torch.matmul(X, Y)
assert torch.allclose(Z_at, Z_matmul)
print("Matrix products agree!\n")

X_elem = X * X

Z_sum = Z_at.sum()
Z_col_mean = Z_at.mean(dim=0)
Z_row_max = Z_at.max(dim=1).values

print("Sum:", Z_sum.item())
print("Col mean:", Z_col_mean)
print("Row max:", Z_row_max)


---
## 2. Broadcasting, indexing, reshaping

Broadcasting lets you operate on tensors of different shapes without writing loops, as long as their trailing dimensions are *compatible* (equal, or one of them is 1).

### Exercise 2.1 — Standardise the columns of a matrix

Given a matrix `M` of shape `(N, D)`, write code that produces `M_std` where each column has mean 0 and standard deviation 1. **Use broadcasting — no explicit loops.**

After your code, the asserts below should pass.

In [ ]:
M = torch.randn(100, 5) * torch.tensor([1., 2., 3., 4., 5.]) + torch.tensor([10., -3., 0., 7., 1.])

M_std = (M - M.mean(dim=0)) / M.std(dim=0, unbiased=False)

assert torch.allclose(M_std.mean(dim=0), torch.zeros(5), atol=1e-5)
assert torch.allclose(M_std.std(dim=0, unbiased=False), torch.ones(5), atol=1e-5)
print('OK')

### Exercise 2.2 — Reshape and permute

Start from `T = torch.arange(24)`.

1. Reshape `T` to shape `(2, 3, 4)` and call it `T1`.
2. Permute `T1`'s axes so the result has shape `(4, 2, 3)`. Call it `T2`.
3. Flatten `T2` back to a 1-D tensor `T3`.
4. Is `T3` equal to `T`? Why or why not?

In [ ]:
T = torch.arange(24)

T1 = T.reshape(2, 3, 4)
T2 = T1.permute(2, 0, 1)
T3 = T2.flatten()

# 4. Is T3 equal to T? 
# No, T3 is NOT equal to T. `permute` changes the memory strides and logical ordering.
# When flattened, the sequence of numbers is entirely different from the original 0 to 23.
print("Is T3 equal to T?", torch.all(T == T3).item())


### Exercise 2.3 — Boolean / fancy indexing

Given `x = torch.randn(1000)`:

1. Build a mask that selects the entries of `x` strictly greater than 1.
2. Replace those entries (in a copy `y`) with the value `1.0`. The original `x` should be untouched.
3. Report the fraction of entries that were clipped.

In [ ]:
x = torch.randn(1000)

mask = x > 1
y = x.clone()
y[mask] = 1.0

frac = mask.float().mean().item()
print(f"Fraction of entries clipped: {frac:.4f}")


---
## 3. Autograd

PyTorch's `autograd` engine records operations on tensors that have `requires_grad=True` into a dynamic computation graph, then computes gradients on demand via `.backward()`.

Key ideas:
- Only **leaf** tensors with `requires_grad=True` accumulate gradients in `.grad`.
- `.backward()` traverses the graph in reverse, multiplying Jacobian-vector products.
- Call `tensor.detach()` to break a tensor out of the graph; wrap code in `torch.no_grad()` to disable tracking entirely.

### Exercise 3.1 — Gradient of a scalar function by hand and by autograd

Let $f(x) = 3x^3 - 5x^2 + 2x - 7$. Analytically, $f'(x) = 9x^2 - 10x + 2$.

1. Create `x = torch.tensor(2.0, requires_grad=True)`.
2. Compute `y = f(x)`, then `y.backward()`.
3. Check that `x.grad` equals $9 \cdot 2^2 - 10 \cdot 2 + 2 = 18$.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = 3*x**3 - 5*x**2 + 2*x - 7

y.backward()

print("Autograd x.grad:", x.grad.item())
print("Analytic gradient:", (9*(2.0**2) - 10*(2.0) + 2))


### Exercise 3.2 — Gradient of a vector-valued function

Let $\mathbf{w} \in \mathbb{R}^3$ and $\mathbf{x} \in \mathbb{R}^3$, and define
$$L = \tfrac{1}{2} \| \mathbf{w} \odot \mathbf{x} - \mathbf{t} \|_2^2$$
for some target `t`.

Compute `dL/dw` for the given numerical values, using autograd. Then check it against the closed-form expression $\nabla_\mathbf{w} L = (\mathbf{w}\odot\mathbf{x} - \mathbf{t}) \odot \mathbf{x}$.

In [ ]:
w = torch.tensor([0.5, -1.0, 2.0], requires_grad=True)
x = torch.tensor([1.0, 2.0, -0.5])
t = torch.tensor([0.0, 0.0, 1.0])

L = 0.5 * torch.sum((w * x - t) ** 2)
L.backward()

print("Autograd dL/dw:", w.grad)

closed_form = (w * x - t) * x
print("Closed form dL/dw:", closed_form)
assert torch.allclose(w.grad, closed_form)


### Exercise 3.3 — `detach`, `no_grad`, and gradient accumulation

Build a small experiment that demonstrates each of these behaviours. For a parameter `p = torch.tensor(1.0, requires_grad=True)`:

1. Compute the gradient of `loss = p**2` twice in a row, **without** calling `p.grad.zero_()` in between. What is `p.grad` after the second call?
2. Repeat, but call `p.grad.zero_()` between the two backward passes.
3. Show that `(p.detach() ** 2).backward()` fails (or produces no gradient).
4. Show that operations inside `with torch.no_grad():` produce tensors with `requires_grad=False`.

In [ ]:
p = torch.tensor(1.0, requires_grad=True)

# 1. Accumulation without zeroing
(p**2).backward()
(p**2).backward()
print("Grad without zeroing:", p.grad.item()) # 2.0 + 2.0 = 4.0

# 2. Zeroing between passes
p.grad.zero_()
(p**2).backward()
print("Grad after zeroing:", p.grad.item()) # 2.0

# 3. Detach failure
try:
    (p.detach() ** 2).backward()
except RuntimeError as e:
    print("Detach error caught: Cannot call backward on detached tensor.")

# 4. no_grad
with torch.no_grad():
    y = p ** 2
    print("Requires grad inside no_grad:", y.requires_grad) # False


---
## 4. Building models with `nn.Module`

Anything trainable in PyTorch is conventionally a subclass of `nn.Module`. You register `nn.Parameter`s (which automatically appear in `.parameters()`) and submodules; you implement `forward(...)`.

Building blocks worth knowing:
- `nn.Linear`, `nn.Conv2d`, `nn.LayerNorm`, `nn.Embedding`
- `nn.ReLU`, `nn.GELU`, `nn.Tanh`
- `nn.Sequential`, `nn.ModuleList`, `nn.ModuleDict`

### Exercise 4.1 — An MLP from scratch (no `nn.Linear`)

Implement a 2-layer MLP **without** using `nn.Linear` — instantiate the weights and biases yourself as `nn.Parameter`s. The model should map `D_in → H → D_out` with a `tanh` non-linearity in between.

Initialise weights with Kaiming-style scaling: `W ~ N(0, 2/fan_in)`.

In [ ]:
class MyMLP(nn.Module):
    def __init__(self, d_in, hidden, d_out):
        super().__init__()
        self.W1 = nn.Parameter(torch.randn(hidden, d_in) * math.sqrt(2.0 / d_in))
        self.b1 = nn.Parameter(torch.zeros(hidden))
        self.W2 = nn.Parameter(torch.randn(d_out, hidden) * math.sqrt(2.0 / hidden))
        self.b2 = nn.Parameter(torch.zeros(d_out))

    def forward(self, x):
        h = x @ self.W1.T + self.b1
        h = torch.tanh(h)
        out = h @ self.W2.T + self.b2
        return out

m = MyMLP(8, 16, 4)
print("Output shape:", m(torch.randn(5, 8)).shape)  # expect torch.Size([5, 4])
print("Parameter count:", sum(p.numel() for p in m.parameters()))


### Exercise 4.2 — The same MLP using `nn.Sequential`

Re-implement the model above using `nn.Linear` layers wrapped in `nn.Sequential`. Confirm that the parameter count matches your hand-built version.

In [ ]:
seq_mlp = nn.Sequential(
    nn.Linear(8, 16),
    nn.Tanh(),
    nn.Linear(16, 4)
)
print("Sequential Parameter count:", sum(p.numel() for p in seq_mlp.parameters()))


### Exercise 4.3 — Parameter counting and `.state_dict()`

Write a function `count_params(model, trainable_only=True)` that returns the total number of parameters. Use it on your MLP.

Then save the model's `state_dict()` to a file and load it back into a fresh instance. Verify the outputs match on a random input.

In [ ]:
def count_params(model, trainable_only=True):
    return sum(p.numel() for p in model.parameters() if not trainable_only or p.requires_grad)

print("MyMLP params via function:", count_params(MyMLP(8, 16, 4)))

# Save and load
torch.save(seq_mlp.state_dict(), 'temp.pt')
new_mlp = nn.Sequential(nn.Linear(8, 16), nn.Tanh(), nn.Linear(16, 4))
new_mlp.load_state_dict(torch.load('temp.pt'))

test_x = torch.randn(5, 8)
assert torch.allclose(seq_mlp(test_x), new_mlp(test_x))
print("State dictionary successfully saved, loaded, and verified.")


---
## 5. Datasets, `DataLoader`, and a training loop

PyTorch separates **datasets** (random-access containers of examples) from **data loaders** (which batch, shuffle, and parallelise). The minimal training loop is:

```python
for epoch in range(num_epochs):
    for xb, yb in loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```

### Exercise 5.1 — A toy regression dataset

Generate a synthetic dataset for the function $y = \sin(2\pi x) + 0.1\,\varepsilon$ with $x \in [0, 1]$ and Gaussian noise $\varepsilon$. Make 1000 training points and 200 validation points. Wrap them in `TensorDataset`s and `DataLoader`s with batch size 32.

In [ ]:
x_train = torch.rand(1000, 1)
y_train = torch.sin(2 * math.pi * x_train) + 0.1 * torch.randn(1000, 1)

x_val = torch.rand(200, 1)
y_val = torch.sin(2 * math.pi * x_val) + 0.1 * torch.randn(200, 1)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=32, shuffle=False)


### Exercise 5.2 — Train an MLP on it

Use your `MyMLP` (or the `nn.Sequential` version) with `d_in=1`, `hidden=64`, `d_out=1`. Train for, say, 200 epochs with `MSELoss` and `torch.optim.Adam(lr=1e-2)`. Log the average training and validation loss per epoch and plot them.

In [ ]:
import matplotlib.pyplot as plt

model = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 1))
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

train_losses, val_losses = [], []

for epoch in range(200):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item() * len(xb)
    train_losses.append(total_loss / len(train_loader.dataset))
    
    model.eval()
    with torch.no_grad():
        val_loss = sum(loss_fn(model(xb), yb).item() * len(xb) for xb, yb in val_loader)
        val_losses.append(val_loss / len(val_loader.dataset))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Training Curve")


### Exercise 5.3 — Plot the fitted curve

On a dense grid of `x` values in `[0, 1]`, predict `y_hat` (under `torch.no_grad()` and in `model.eval()` mode), and plot the predictions against the true sine and the noisy training points.

In [ ]:
plt.subplot(1, 2, 2)
model.eval()
grid = torch.linspace(0, 1, 100).unsqueeze(1)
with torch.no_grad():
    preds = model(grid)

plt.scatter(x_train, y_train, s=1, alpha=0.5, label="Data")
plt.plot(grid, preds, color='red', label="Fitted")
plt.plot(grid, torch.sin(2 * math.pi * grid), color='green', linestyle='--', label="True")
plt.legend()
plt.title("Model Fit")
plt.show()


---
## 6. Custom loss functions

A loss function is just a differentiable function from `(prediction, target)` to a scalar. You can write it as a plain Python function or as an `nn.Module` (handy when it has learnable parameters or state).

### Exercise 6.1 — Huber loss

Implement the Huber loss
$$\ell_\delta(r) = \begin{cases} \tfrac{1}{2} r^2 & |r| \le \delta \\ \delta\,(|r| - \tfrac{1}{2}\delta) & |r| > \delta \end{cases}$$
where $r = \hat{y} - y$. Take the **mean** over the batch.

Sanity check: for very small residuals it should agree with $\tfrac{1}{2}\text{MSE}$; for very large residuals it should grow linearly. Compare against `nn.SmoothL1Loss(beta=delta)` (note the small difference: `SmoothL1` uses $r^2/(2\delta)$ on the inside, not $r^2/2$).

In [ ]:
def huber_loss(pred, target, delta=1.0):
    r = torch.abs(pred - target)
    loss = torch.where(r <= delta, 0.5 * r**2, delta * (r - 0.5 * delta))
    return loss.mean()


### Exercise 6.2 — Cross-entropy from scratch (numerically stable)

Implement multi-class cross-entropy from raw logits *without* calling `F.cross_entropy`, `F.log_softmax`, or `nn.LogSoftmax`.

Use the log-sum-exp trick:
$$\log\sum_j e^{z_j} = m + \log\sum_j e^{z_j - m}, \quad m = \max_j z_j$$

Verify your result against `F.cross_entropy` to within `1e-6`.

In [ ]:
def my_cross_entropy(logits, targets):
    """logits: (N, C); targets: (N,) long class indices. Returns scalar mean loss."""
    m = logits.max(dim=1, keepdim=True).values
    lse = m.squeeze() + torch.log(torch.sum(torch.exp(logits - m), dim=1))
    target_logits = logits[torch.arange(logits.size(0)), targets]
    return (lse - target_logits).mean()

# Smoke test
logits = torch.randn(8, 5)
targets = torch.randint(0, 5, (8,))
assert torch.allclose(my_cross_entropy(logits, targets), F.cross_entropy(logits, targets), atol=1e-6)
print('Cross-entropy OK')


---
## 7. Custom optimizers

A PyTorch optimizer is a class with a `.step()` method that, after `loss.backward()` has populated `p.grad`, updates each parameter `p` in place. We will write four optimizers from scratch.

The skeleton looks like:

```python
class MyOpt:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr
        self.state = {id(p): {} for p in self.params}

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            # update p in place

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()
```

All updates **must** happen under `torch.no_grad()` (or use `p.data` / `p.add_`) so we don't track them in the graph.

### Exercise 7.1 — Plain SGD

Implement `MySGD` with the update
$$\theta \leftarrow \theta - \eta\, g.$$

In [ ]:
class MySGD:
    def __init__(self, params, lr=1e-2):
        self.params = list(params)
        self.lr = lr

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


### Exercise 7.2 — SGD with momentum (Polyak)

Implement the heavy-ball update:
$$v \leftarrow \mu\, v + g, \qquad \theta \leftarrow \theta - \eta\, v.$$
Initialise `v` to zeros the first time you see each parameter.

In [ ]:
class MyMomentum:
    def __init__(self, params, lr=1e-2, momentum=0.9):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.state = {id(p): {'v': torch.zeros_like(p)} for p in self.params}

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                v = self.state[id(p)]['v']
                v.mul_(self.momentum).add_(p.grad)
                p.sub_(self.lr * v)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None: p.grad.zero_()


### Exercise 7.3 — RMSProp

Implement RMSProp:
$$s \leftarrow \alpha\, s + (1-\alpha)\, g^2, \qquad \theta \leftarrow \theta - \frac{\eta}{\sqrt{s} + \epsilon}\, g.$$

In [ ]:
class MyRMSProp:
    def __init__(self, params, lr=1e-3, alpha=0.99, eps=1e-8):
        self.params = list(params)
        self.lr, self.alpha, self.eps = lr, alpha, eps
        self.state = {id(p): {'s': torch.zeros_like(p)} for p in self.params}

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                s = self.state[id(p)]['s']
                s.mul_(self.alpha).addcmul_(p.grad, p.grad, value=1 - self.alpha)
                p.addcdiv_(p.grad, s.sqrt() + self.eps, value=-self.lr)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None: p.grad.zero_()


### Exercise 7.4 — Adam

Implement Adam with bias correction:
$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
$$\hat m_t = m_t / (1 - \beta_1^t), \quad \hat v_t = v_t / (1 - \beta_2^t)$$
$$\theta_t = \theta_{t-1} - \eta\, \hat m_t / (\sqrt{\hat v_t} + \epsilon)$$

Track the step count $t$ per parameter (it is the same for all, but you'll want it stored).

In [ ]:
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr, self.beta1, self.beta2, self.eps = lr, betas[0], betas[1], eps
        self.state = {id(p): {'m': torch.zeros_like(p), 'v': torch.zeros_like(p), 't': 0} for p in self.params}

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is None: continue
            state = self.state[id(p)]
            state['t'] += 1
            m, v, t = state['m'], state['v'], state['t']
            
            m.mul_(self.beta1).add_(p.grad, alpha=1 - self.beta1)
            v.mul_(self.beta2).addcmul_(p.grad, p.grad, value=1 - self.beta2)
            
            m_hat = m / (1 - self.beta1**t)
            v_hat = v / (1 - self.beta2**t)
            
            p.addcdiv_(m_hat, v_hat.sqrt() + self.eps, value=-self.lr)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None: p.grad.zero_()


### Exercise 7.5 — Bake-off on the toy regression task

Train your sine-fitting MLP **four times** with identical initial weights, using `MySGD`, `MyMomentum`, `MyRMSProp`, and `MyAdam`. Plot the training-loss curve for each on the same axes. Comment on what you observe.

*Hint:* use `copy.deepcopy` on the freshly-initialised model so each optimizer starts from the same weights.

In [ ]:
import copy
base_model = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 1))
opts = {
    'SGD': MySGD,
    'Momentum': MyMomentum,
    'RMSProp': MyRMSProp,
    'Adam': MyAdam
}

plt.figure(figsize=(10, 6))
for name, OptClass in opts.items():
    m = copy.deepcopy(base_model)
    opt = OptClass(m.parameters())
    
    losses = []
    for epoch in range(50):
        total_loss = 0
        for xb, yb in train_loader:
            loss = nn.MSELoss()(m(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(xb)
        losses.append(total_loss / len(train_loader.dataset))
    plt.plot(losses, label=name)

plt.legend()
plt.title("Optimizer Bake-off")
plt.show()


### Exercise 7.6 — Weight decay, properly

Add a `weight_decay` parameter to `MyAdam`. There are **two** common ways to apply it:

1. **L2 regularisation** — add `weight_decay * p` to the gradient *before* the moment updates.
2. **Decoupled weight decay (AdamW)** — apply `p -= lr * weight_decay * p` directly to the parameter, *separately* from the adaptive update.

Implement both as options. Discuss in a short comment why decoupled weight decay tends to behave better with Adam-family optimizers.

In [ ]:
class MyAdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay
        self.state = {id(p): {'m': torch.zeros_like(p), 'v': torch.zeros_like(p), 't': 0} for p in self.params}

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is None: continue
            
            # Decoupled weight decay (AdamW strategy)
            p.mul_(1.0 - self.lr * self.weight_decay)
            
            state = self.state[id(p)]
            state['t'] += 1
            m, v, t = state['m'], state['v'], state['t']
            
            m.mul_(self.beta1).add_(p.grad, alpha=1 - self.beta1)
            v.mul_(self.beta2).addcmul_(p.grad, p.grad, value=1 - self.beta2)
            
            m_hat = m / (1 - self.beta1**t)
            v_hat = v / (1 - self.beta2**t)
            
            p.addcdiv_(m_hat, v_hat.sqrt() + self.eps, value=-self.lr)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None: p.grad.zero_()

# Discussion:
# Decoupled weight decay (AdamW) applies the decay directly to the parameter: `p -= lr * weight_decay * p`.
# L2 Regularization (adding `weight_decay * p` to the gradient) gets scaled by Adam's adaptive 
# learning rates (m / sqrt(v)). This means variables with large historical gradients get less regularization.
# Decoupled decay ensures all weights decay uniformly regardless of their gradient history.


---
## 8. Custom autograd functions

Sometimes you need to define an operation whose forward or backward pass autograd doesn't already know about — e.g. a non-standard non-linearity, a discrete operation with a straight-through gradient, or a numerically tricky composite that you can simplify by hand.

The recipe is to subclass `torch.autograd.Function`:

```python
class MyFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, *inputs):
        ctx.save_for_backward(...)
        return output

    @staticmethod
    def backward(ctx, grad_output):
        (...) = ctx.saved_tensors
        return grad_inputs  # one per input to forward(), in the same order
```

Use `torch.autograd.gradcheck` (with `dtype=torch.float64`) to verify your backward is correct.

### Exercise 8.1 — `MySquare`

Warm-up. Implement `y = x**2` as a `torch.autograd.Function`. The derivative is `2x`, so the backward should return `grad_output * 2 * x`.

Verify with `gradcheck`.

In [ ]:
class MySquare(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x ** 2

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        return grad_output * 2 * x

x = torch.randn(4, dtype=torch.float64, requires_grad=True)
torch.autograd.gradcheck(MySquare.apply, (x,))


### Exercise 8.2 — `MyReLU`

Implement ReLU as a custom autograd function. The derivative is 1 for `x > 0` and 0 otherwise (the choice at `x = 0` is a sub-gradient — 0 is the conventional pick).

Then verify with `gradcheck` (perturb only at non-zero points).

In [ ]:
class MyReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        return grad_output * (x > 0).float()


### Exercise 8.3 — Numerically stable `MyLogSoftmax`

Implement log-softmax along a chosen dimension, with a hand-written backward.

Forward (use log-sum-exp for stability):
$$y_i = z_i - \log\sum_j e^{z_j}.$$

Backward — given upstream `g = ∂L/∂y`, the Jacobian of log-softmax is $I - \mathbf{1}\,\mathrm{softmax}(z)^\top$ along the chosen axis, so:
$$\frac{\partial L}{\partial z_i} = g_i - \left(\sum_j g_j\right) \cdot \mathrm{softmax}(z)_i.$$

Verify against `F.log_softmax` both forward (values) and backward (gradients) on a random input.

In [ ]:
class MyLogSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(ctx, z, dim):
        m, _ = z.max(dim=dim, keepdim=True)
        z_norm = z - m
        log_sum_exp = torch.log(torch.sum(torch.exp(z_norm), dim=dim, keepdim=True))
        out = z_norm - log_sum_exp
        ctx.save_for_backward(out)
        ctx.dim = dim
        return out

    @staticmethod
    def backward(ctx, grad_output):
        out, = ctx.saved_tensors
        dim = ctx.dim
        softmax_z = torch.exp(out)
        grad_z = grad_output - grad_output.sum(dim=dim, keepdim=True) * softmax_z
        return grad_z, None


### Exercise 8.4 — Straight-through estimator

Implement `MyRound` whose forward is `torch.round(x)` (non-differentiable), but whose backward acts as the identity — i.e. `grad_x = grad_output`. This is the *straight-through estimator*, commonly used to backprop through discrete operations (quantisation, hard-attention, etc.).

Test that a tiny linear layer trained with `MyRound` in the middle still learns to fit `y = 3x + 1` on a small dataset.

In [ ]:
class MyRound(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return torch.round(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output


### Exercise 8.5 — A composite op with a custom backward: `MySwish`

Swish is $f(x) = x \cdot \sigma(x)$, where $\sigma$ is the logistic sigmoid. Its derivative is
$$f'(x) = \sigma(x) + x\,\sigma(x)\,(1 - \sigma(x)) = f(x) + \sigma(x)\,(1 - f(x)).$$

Implement it as a `torch.autograd.Function` that saves only `sigmoid(x)` from the forward (so the backward is cheaper — no recompute). Verify with `gradcheck` and compare runtime against the naïve `x * torch.sigmoid(x)` version on a large tensor.

In [ ]:
class MySwish(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        sig_x = torch.sigmoid(x)
        ctx.save_for_backward(x, sig_x)
        return x * sig_x

    @staticmethod
    def backward(ctx, grad_output):
        x, sig_x = ctx.saved_tensors
        f_x = x * sig_x
        grad_in = f_x + sig_x * (1 - f_x)
        return grad_output * grad_in


---
## 9. Hooks, gradient surgery, higher-order gradients

A few more advanced tools.

### Exercise 9.1 — Register a backward hook to log gradient norms

Pick any of your `nn.Linear` layers and use `module.register_full_backward_hook(...)` to print the L2 norm of the gradient flowing through it on each backward pass. Train for a few iterations and observe.

In [ ]:
def hook_fn(module, grad_input, grad_output):
    print(f"Hook fired! Grad norm: {grad_output[0].norm().item():.4f}")

layer = nn.Linear(10, 5)
layer.register_full_backward_hook(hook_fn)

x = torch.randn(2, 10)
layer(x).sum().backward()


### Exercise 9.2 — Gradient clipping by hand

Implement global-norm gradient clipping yourself (don't call `torch.nn.utils.clip_grad_norm_`). The recipe: compute `total_norm = sqrt(sum(g.pow(2).sum() for g in grads))`; if `total_norm > max_norm`, scale every gradient by `max_norm / (total_norm + 1e-6)`.

In [ ]:
def clip_grad_norm(params, max_norm):
    params = [p for p in params if p.grad is not None]
    total_norm = torch.sqrt(sum(p.grad.pow(2).sum() for p in params))
    if total_norm > max_norm:
        scale = max_norm / (total_norm + 1e-6)
        for p in params:
            p.grad.mul_(scale)


### Exercise 9.3 — Second-order gradients

For $f(x, y) = x^2 y + y^3$ at the point $(1, 2)$, compute:

1. $\partial f / \partial x$ and $\partial f / \partial y$ using `torch.autograd.grad(..., create_graph=True)`.
2. The full $2\times 2$ Hessian matrix at $(1, 2)$ by differentiating each first-order gradient again.

Check against the closed form: $\nabla f = (2xy, x^2 + 3y^2)$ and Hessian = $\begin{pmatrix} 2y & 2x \\ 2x & 6y \end{pmatrix}$, which at $(1,2)$ is $\begin{pmatrix} 4 & 2 \\ 2 & 12 \end{pmatrix}$.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)

f = x**2 * y + y**3

df_dx, df_dy = torch.autograd.grad(f, (x, y), create_graph=True)

d2f_dx2, d2f_dxdy = torch.autograd.grad(df_dx, (x, y), retain_graph=True)
d2f_dydx, d2f_dy2 = torch.autograd.grad(df_dy, (x, y))

Hessian = torch.tensor([
    [d2f_dx2, d2f_dxdy],
    [d2f_dydx, d2f_dy2]
])
print("Hessian at (1, 2):\n", Hessian)


---
## 10. Putting it all together

Build a small classifier on a real toy dataset using **your own** components throughout. This is open-ended — try to use as many of the pieces you wrote above as you can.

### Exercise 10 — Two-moons classification with your stack

1. Generate a `make_moons`-style dataset (you can use `sklearn.datasets.make_moons` or roll your own).
2. Build an MLP classifier (1 hidden layer of 32 units is enough). Use `MyReLU` or `MySwish` as the activation.
3. Train it with **your** `MyAdam`, using **your** `my_cross_entropy` as the loss.
4. Apply your hand-written gradient-norm clipping at every step with `max_norm=1.0`.
5. Plot the decision boundary, the training-loss curve, and final accuracy.

If everything is wired together correctly you should comfortably hit > 95% accuracy.

In [ ]:
from sklearn.datasets import make_moons

# 1. Dataset
X_np, y_np = make_moons(n_samples=1000, noise=0.1, random_state=0)
X_t = torch.tensor(X_np, dtype=torch.float32)
y_t = torch.tensor(y_np, dtype=torch.long)
loader = DataLoader(TensorDataset(X_t, y_t), batch_size=32, shuffle=True)

# 2. Model using custom components
class MoonMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 32)
        self.fc2 = nn.Linear(32, 2)
    def forward(self, x):
        return self.fc2(MyReLU.apply(self.fc1(x)))

model = MoonMLP()

# 3. Train using custom optim and loss
opt = MyAdam(model.parameters(), lr=1e-2)

for epoch in range(50):
    for xb, yb in loader:
        pred = model(xb)
        loss = my_cross_entropy(pred, yb)
        
        opt.zero_grad()
        loss.backward()
        
        # 4. Apply custom clipping
        clip_grad_norm(model.parameters(), max_norm=1.0)
        opt.step()

# Evaluate
with torch.no_grad():
    acc = (model(X_t).argmax(dim=1) == y_t).float().mean()
print(f"Final Accuracy: {acc.item() * 100:.2f}%")


---
## Where to go next

If you finished the above:
- Re-do Exercise 7.4 (Adam) so that it supports **parameter groups** (different `lr` per group), like real `torch.optim` optimizers.
- Write a custom autograd function for **batch-norm** (forward + backward by hand) and compare against `nn.BatchNorm1d`.
- Read the source of `torch.optim.Adam` and `torch.nn.functional.cross_entropy` and compare them to your implementations.
- Look at `torch.func` (functorch) — `grad`, `vmap`, `jacrev` — and re-do the Hessian exercise in one line.